# 04 — Zero-Shot Humanitarian Classification (Qwen2.5-VL-7B-Instruct)

Run zero-shot classification on the **humanitarian** task (5 classes) across all 3 modalities and both train/test splits.

| # | Modality | Split | Output |
|---|----------|-------|--------|
| 1 | text_only | test | `results/zeroshot/qwen2.5-vl-7b/humanitarian/text_only/test/` |
| 2 | text_only | train | `results/zeroshot/qwen2.5-vl-7b/humanitarian/text_only/train/` |
| 3 | image_only | test | `results/zeroshot/qwen2.5-vl-7b/humanitarian/image_only/test/` |
| 4 | image_only | train | `results/zeroshot/qwen2.5-vl-7b/humanitarian/image_only/train/` |
| 5 | text_image | test | `results/zeroshot/qwen2.5-vl-7b/humanitarian/text_image/test/` |
| 6 | text_image | train | `results/zeroshot/qwen2.5-vl-7b/humanitarian/text_image/train/` |

## Setup

In [ ]:
import sys, os, gc
sys.path.insert(0, os.path.abspath(".."))

import torch
from scripts.zeroshot_qwen import load_model, run_zeroshot

# Configuration
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
TASK = "humanitarian"
DATA_ROOT = os.path.abspath("../data")
OUTPUT_DIR = os.path.abspath("../results/zeroshot")

# Helper: clean up predictions and free GPU cache between experiments
def cleanup(*vars_to_del):
    """Delete prediction variables and free GPU memory."""
    for v in vars_to_del:
        del v
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Load model once — reused across all 6 experiments
model, processor = load_model(MODEL_ID)

## 1. Text-Only — Test Set

In [ ]:
preds, metrics_text_test = run_zeroshot(
    task=TASK, modality="text_only", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 2. Text-Only — Train Set

In [ ]:
preds, metrics_text_train = run_zeroshot(
    task=TASK, modality="text_only", split="train",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 3. Image-Only — Test Set

In [ ]:
preds, metrics_img_test = run_zeroshot(
    task=TASK, modality="image_only", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 4. Image-Only — Train Set

In [ ]:
preds, metrics_img_train = run_zeroshot(
    task=TASK, modality="image_only", split="train",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 5. Text+Image — Test Set

In [ ]:
preds, metrics_ti_test = run_zeroshot(
    task=TASK, modality="text_image", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 6. Text+Image — Train Set

In [ ]:
preds, metrics_ti_train = run_zeroshot(
    task=TASK, modality="text_image", split="train",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## Summary — All 6 Experiments

In [ ]:
import pandas as pd

all_metrics = [
    metrics_text_test, metrics_text_train,
    metrics_img_test, metrics_img_train,
    metrics_ti_test, metrics_ti_train,
]

summary = pd.DataFrame([
    {
        "Modality": m["modality"],
        "Split": m["split"],
        "Samples": m["num_samples"],
        "Accuracy": f"{m['accuracy']:.4f}",
        "Weighted F1": f"{m['weighted_f1']:.4f}",
        "Macro F1": f"{m['macro_f1']:.4f}",
        "Unparseable": m["num_unparseable"],
    }
    for m in all_metrics
])

print(f"Task: {TASK}")
print(f"Model: {MODEL_ID}\n")
display(summary)

## Cleanup — Free GPU Memory

In [ ]:
del model, processor
del metrics_text_test, metrics_text_train
del metrics_img_test, metrics_img_train
del metrics_ti_test, metrics_ti_train

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.1f} GB")
print("\nNote: If memory is still high, restart the kernel to fully release GPU memory.")